# 🎯 Vision Engine — Auto-Labeling Pipeline
**GroundingDINO-Tiny on free T4 GPU**

### Steps
1. Mount Google Drive
2. Upload your `datasets/raw/` zip to Drive first
3. Run this notebook — labels are saved back to Drive
4. Download the labels zip

**Runtime:** ~25 minutes for 1,237 images on T4 GPU

---
**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Cell 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
!pip install -q transformers torch torchvision tqdm pillow
print('Done.')

In [ ]:
# ── Cell 3: Config — SET THESE PATHS ────────────────────────────────────────
import os

# Path to the raw images zip you uploaded to Google Drive
# Upload: D:\VISION\datasets\raw\ zipped as 'vision_raw.zip' to your Drive
DRIVE_ZIP_PATH   = '/content/drive/MyDrive/vision_raw.zip'   # ← CHANGE IF NEEDED

# Where to extract images
RAW_DIR          = '/content/vision_raw'

# Where to save labels
LABELS_DIR       = '/content/vision_labels'

# Drive output path for the labels zip
DRIVE_OUTPUT_ZIP = '/content/drive/MyDrive/vision_labels.zip' # ← labels saved here

# Detection settings
CONF_THRESHOLD   = 0.30
CLASSES = {
    'bubble':    'speech bubble . dialogue bubble . thought bubble',
    'narration': 'narration box . caption box . text box',
}

ALL_CLASSES = ['bubble', 'narration', 'text', 'sfx', 'panel']

os.makedirs(LABELS_DIR, exist_ok=True)
print('Config set.')

In [ ]:
# ── Cell 4: Extract images ───────────────────────────────────────────────────
import zipfile

if os.path.exists(DRIVE_ZIP_PATH):
    print(f'Extracting {DRIVE_ZIP_PATH} ...')
    with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
        z.extractall(RAW_DIR)
    print('Extraction complete.')
else:
    print(f'[ERROR] Zip not found at {DRIVE_ZIP_PATH}')
    print('Please upload vision_raw.zip to your Google Drive first.')

# Count images
from pathlib import Path
all_images = sorted(Path(RAW_DIR).rglob('*.jpg')) + sorted(Path(RAW_DIR).rglob('*.png'))
print(f'Total images found: {len(all_images)}')

In [ ]:
# ── Cell 5: Load GroundingDINO-Tiny ─────────────────────────────────────────
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection

MODEL_ID = 'IDEA-Research/grounding-dino-tiny'
device   = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {device}')
if device == 'cpu':
    print('[WARNING] No GPU detected — make sure Runtime → T4 GPU is selected!')

print('Loading model...')
processor = AutoProcessor.from_pretrained(MODEL_ID)
model     = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID).to(device)
model.eval()
print('Model ready.')

In [ ]:
# ── Cell 6: Auto-label all images ───────────────────────────────────────────
import json
import time
import warnings
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings('ignore')  # suppress FutureWarning

# Build text prompt
text_prompt = ' . '.join(CLASSES.values()) + ' .'

# Build label map: phrase -> class name
label_map = {}
for cls, prompt in CLASSES.items():
    for phrase in prompt.split(' . '):
        label_map[phrase.strip()] = cls

class_to_idx = {c: i for i, c in enumerate(ALL_CLASSES)}

def bbox_to_yolo(box, img_w, img_h):
    x_min, y_min, x_max, y_max = box
    cx = max(0.0, min(1.0, (x_min + x_max) / 2 / img_w))
    cy = max(0.0, min(1.0, (y_min + y_max) / 2 / img_h))
    w  = max(0.0, min(1.0, (x_max - x_min) / img_w))
    h  = max(0.0, min(1.0, (y_max - y_min) / img_h))
    return cx, cy, w, h

total_det = 0
errors    = 0
t_start   = time.time()

PROGRESS_FILE = '/content/autolabel_progress.json'

# Resume support
done = set()
if os.path.exists(PROGRESS_FILE):
    done = set(json.loads(open(PROGRESS_FILE).read()).get('done', []))
    print(f'Resuming — {len(done)} images already done.')

to_process = [p for p in all_images if str(p) not in done]
print(f'Processing {len(to_process)} images...\n')

for img_path in tqdm(to_process, unit='img'):
    try:
        image = Image.open(img_path).convert('RGB')
        img_w, img_h = image.size

        inputs = processor(images=image, text=text_prompt, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            threshold=CONF_THRESHOLD,
            text_threshold=CONF_THRESHOLD,
            target_sizes=[(img_h, img_w)],
        )[0]

        label_lines = []
        text_labels = results.get('text_labels', results.get('labels', []))
        for box, score, label_text in zip(results['boxes'].tolist(), results['scores'].tolist(), text_labels):
            matched = None
            for phrase, cls in label_map.items():
                if phrase.lower() in str(label_text).lower() or str(label_text).lower() in phrase.lower():
                    matched = cls
                    break
            if matched is None:
                continue
            cx, cy, w, h = bbox_to_yolo(box, img_w, img_h)
            label_lines.append(f'{class_to_idx[matched]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

        label_file = os.path.join(LABELS_DIR, img_path.stem + '.txt')
        with open(label_file, 'w') as f:
            f.write('\n'.join(label_lines))

        total_det += len(label_lines)
        done.add(str(img_path))

        if len(done) % 100 == 0:
            with open(PROGRESS_FILE, 'w') as f:
                json.dump({'done': sorted(done)}, f)

    except Exception as e:
        print(f'\n[ERROR] {img_path.name}: {e}')
        errors += 1

# Save final progress
with open(PROGRESS_FILE, 'w') as f:
    json.dump({'done': sorted(done)}, f)

elapsed = time.time() - t_start
print(f'\nDone! {len(to_process)} images in {elapsed/60:.1f} min')
print(f'Total detections: {total_det} | Errors: {errors}')

In [ ]:
# ── Cell 7: Stats ───────────────────────────────────────────────────────────
from collections import defaultdict

ALL_CLASSES = ['bubble', 'narration', 'text', 'sfx', 'panel']
label_files = list(Path(LABELS_DIR).glob('*.txt'))
class_counts = defaultdict(int)
empty = 0

for lf in label_files:
    lines = lf.read_text().strip().splitlines()
    if not lines:
        empty += 1
        continue
    for line in lines:
        parts = line.split()
        if parts:
            class_counts[ALL_CLASSES[int(parts[0])]] += 1

print(f'Label files: {len(label_files)}')
print(f'Empty (no detection): {empty}')
print('\nClass distribution:')
total = sum(class_counts.values())
for cls in ALL_CLASSES:
    n = class_counts.get(cls, 0)
    bar = '█' * int(n / max(total, 1) * 40)
    print(f'  {cls:<12} {n:>6}  {bar}')
print(f'  {"TOTAL":<12} {total:>6}')

In [ ]:
# ── Cell 8: Zip labels and save to Drive ────────────────────────────────────
import zipfile

OUTPUT_ZIP = '/content/vision_labels.zip'
label_files = list(Path(LABELS_DIR).glob('*.txt'))

print(f'Zipping {len(label_files)} label files...')
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for lf in label_files:
        z.write(lf, lf.name)

# Copy to Google Drive
import shutil
shutil.copy(OUTPUT_ZIP, DRIVE_OUTPUT_ZIP)

size_mb = os.path.getsize(OUTPUT_ZIP) / 1024 / 1024
print(f'\nSaved to Drive: {DRIVE_OUTPUT_ZIP}')
print(f'Size: {size_mb:.1f} MB')
print('\nNext: Download vision_labels.zip from Drive,')
print('then extract into D:\\VISION\\datasets\\annotated\\v1.0\\labels\\')